In [ ]:
import os

if "COLAB_GPU" in os.environ:
  print('ok')
  # !pip install -U torch # requires torch 2.1.1+ (for efficient sdpa implementation)
  !pip install PyMuPDF
  !pip install sentence-transformers # for embedding models
  !pip install tqdm
  !pip install accelerate
  !pip install bitsandbytes
  !pip install flash-attn --no-build-isolation

ok
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 20.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 96.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=253780426 sha256=4e2f9e39313266b1544b68138b15b91ee6221eccf14f7902b7c6620351340810
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn


In [ ]:
os.environ # to show colab information

environ{'SHELL': '/bin/bash',
        'NV_LIBCUBLAS_VERSION': '12.5.3.2-1',
        'NVIDIA_VISIBLE_DEVICES': 'all',
        'COLAB_JUPYTER_TRANSPORT': 'ipc',
        'NV_NVML_DEV_VERSION': '12.5.82-1',
        'NV_CUDNN_PACKAGE_NAME': 'libcudnn9-cuda-12',
        'CGROUP_MEMORY_EVENTS': '/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events',
        'NV_LIBNCCL_DEV_PACKAGE': 'libnccl-dev=2.22.3-1+cuda12.5',
        'NV_LIBNCCL_DEV_PACKAGE_VERSION': '2.22.3-1',
        'VM_GCE_METADATA_HOST': '169.254.169.253',
        'MODEL_PROXY_HOST': 'https://mp.kaggle.net',
        'HOSTNAME': '90fb97d35135',
        'LANGUAGE': 'en_US',
        'TBE_RUNTIME_ADDR': '172.28.0.1:8011',
        'GCE_METADATA_TIMEOUT': '3',
        'NVIDIA_REQUIRE_CUDA': 'cuda>=12.5 brand=unknown,driver>=470,driver<471 brand=grid,driver>=470,driver<471 brand=tesla,driver>=470,driver<471 brand=nvidia,driver>=470,driver<471 brand=quadro,driver>=470,driver<471 brand=quadrortx,driver>=470,driver<

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers sentence-transformers

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: sentence-transformers 5.2.1
Uninstalling sentence-transformers-5.2.1:
  Successfully uninstalled sentence-transformers-5.2.1
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 107.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 90.5 MB/s eta 0:00:00
     ━━━━

## Download the Docs

In [ ]:
import os
import requests

pdf_path = "ML.pdf"

# Download pdf if doesn`t exits
if not os.path.exists(pdf_path):
  print("File does not exist")

  # The URL of the PDF of Download
  url = 'https://shashwatwork.github.io/assets/files/ml_ebook.pdf'

  filename=pdf_path

  # send the GET request to download the PDF

  response = requests.get(url)

  # check if the request was successful
  if response.status_code == 200:
    # open a file in binary write mode and save the contant it
    with open(filename, 'wb') as file:
      file.write(response.content)
    print(f'write the file has been downloaded and saved as {filename}')
  else:
    print(f'the file failed to download status code: {response.status_code}')
else:
  print(f"file {pdf_path} exists")


File does not exist
write the file has been downloaded and saved as ML.pdf


## Extract the text from documents

In [ ]:
from tqdm.auto import tqdm
import fitz

def text_formater(text:str) -> str:
  """ Perform the minner formatting """
  cleaned_text=text.replace("\n", " ").strip()

  return cleaned_text
# Note: focus on only text rather than images and table

def open_and_read_pdf(pdf_path:str)->list[dict]:
  """
  Open : the pdf and read the text page by page

  Parameter get function:
    pdf_path the file path your pdf to open and read the document
  Return:
    list of and dictionary fromat [{}, {}] to return this function

  """
  docs = fitz.open(pdf_path)
  page_and_text=[]

  for page_number , page in tqdm(enumerate(docs)):
    text=page.get_text()
    text=text_formater(text)
    page_and_text.append({
        "page_number": page_number,
        "page_total_words": len(text.split(' ')),
        "page_total_char": len(text),
        "page_total_sentence": len(text.split('. ')),
        "page_total_token": len(text) /4,
        "text": text
    })

  return page_and_text


page_and_text = open_and_read_pdf(pdf_path=pdf_path)
page_and_text[:20]




0it [00:00, ?it/s]

[{'page_number': 0,
  'page_total_words': 1,
  'page_total_char': 0,
  'page_total_sentence': 1,
  'page_total_token': 0.0,
  'text': ''},
 {'page_number': 1,
  'page_total_words': 1,
  'page_total_char': 0,
  'page_total_sentence': 1,
  'page_total_token': 0.0,
  'text': ''},
 {'page_number': 2,
  'page_total_words': 30,
  'page_total_char': 237,
  'page_total_sentence': 1,
  'page_total_token': 59.25,
  'text': 'Aurélien Géron Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow Concepts, Tools, and Techniques to Build Intelligent Systems SECOND EDITION Boston Farnham Sebastopol Tokyo Beijing Boston Farnham Sebastopol Tokyo Beijing'},
 {'page_number': 3,
  'page_total_words': 249,
  'page_total_char': 1777,
  'page_total_sentence': 14,
  'page_total_token': 444.25,
  'text': '978-1-492-03264-9 [LSI] Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow by Aurélien Géron Copyright © 2019 O’Reilly Media. All rights reserved. Printed in the United States of Am

In [ ]:
import random
random_Sam= random.sample(page_and_text, k=5)

In [ ]:
random_Sam

[{'page_number': 161,
  'page_total_words': 271,
  'page_total_char': 1719,
  'page_total_sentence': 10,
  'page_total_token': 429.75,
  'text': 'import numpy as np from sklearn import datasets from sklearn.pipeline import Pipeline from sklearn.preprocessing import StandardScaler from sklearn.svm import LinearSVC iris = datasets.load_iris() X = iris["data"][:, (2, 3)]  # petal length, petal width y = (iris["target"] == 2).astype(np.float64)  # Iris-Virginica svm_clf = Pipeline([         ("scaler", StandardScaler()),         ("linear_svc", LinearSVC(C=1, loss="hinge")),     ]) svm_clf.fit(X, y) Then, as usual, you can use the model to make predictions: >>> svm_clf.predict([[5.5, 1.7]]) array([1.]) Unlike Logistic Regression classifiers, SVM classifiers do not out‐ put probabilities for each class. Alternatively, you could use the SVC class, using SVC(kernel="linear", C=1), but it is much slower, especially with large training sets, so it is not recommended. Another option is to use the 

In [ ]:
import pandas as pd

df=pd.DataFrame(page_and_text)

In [ ]:
df

,page_number,page_total_words,page_total_char,page_total_sentence,page_total_token,text
0,0,1,0,1,0.00,
1,1,1,0,1,0.00,
2,2,30,237,1,59.25,Aurélien Géron Hands-on Machine Learning with ...
3,3,249,1777,14,444.25,978-1-492-03264-9 [LSI] Hands-on Machine Learn...
4,4,2467,3246,90,811.50,Table of Contents 1. The Machine Learning Land...
...,...,...,...,...,...,...
274,274,190,1131,10,282.75,Figure 9-22. Bayesian Gaussian mixture model P...
275,275,350,1771,13,442.75,Bayes’ theorem (Equation 9-2) tells us how to ...
276,276,363,2263,15,565.75,"In practice, there are different techniques to..."
277,277,440,2785,19,696.25,Other Anomaly Detection and Novelty Detection ...


In [ ]:
df.describe()

,page_number,page_total_words,page_total_char,page_total_sentence,page_total_token
count,279.000000,279.000000,279.000000,279.000000,279.000000
mean,139.000000,335.738351,1750.713262,13.591398,437.678315
std,80.684571,408.443051,691.596170,12.272303,172.899043
min,0.000000,1.000000,0.000000,1.000000,0.000000
25%,69.500000,222.000000,1367.500000,9.000000,341.875000
50%,139.000000,302.000000,1786.000000,12.000000,446.500000
75%,208.500000,356.500000,2124.500000,15.000000,531.125000
max,278.000000,3948.000000,4918.000000,124.000000,1229.500000
